# F4 Grobner Basis Algorithm In Python with Sympy by [Murage Kibicho](https://x.com/murage_kibicho)
We code the F4 algorithm for our Grobner basis solver following (Faugère, 1999)

This guide is best followed alongside this [LeetArxiv article](https://leetarxiv.substack.com/p/f4-algorithm-for-grobner-basis-reduction).
<div align="center">
<img src="https://github.com/MurageKibicho/F4-Grobner-Basis-Algorithm-In-Python-with-Sympy/blob/main/Abstract.png?raw=true:" alt="Faugerre Paper" width=400>
</div>





#Sympy's implementation

In [ ]:
import sympy as sp
import time
#Define variables
x, y, z = sp.symbols('x y z')

#Define the polynomials
f1 = x**3 + y**2 + x*z - 1
f2 = x**2 + y**2 + z - 1
f3 = y**2*z + x*z**2 - 1

#Compute the reduced Groebner basis
startTime = time.perf_counter()
G = sp.groebner([f1, f2, f3],x, y, z, order='grevlex',domain=sp.GF(101),method='buchberger')
endTime = time.perf_counter()
elapsedTime = endTime - startTime
print(f"Reduced Groebner has {len(G.polys)} basis polynomials:")
for g in G.polys:
  print(g.as_expr())

print(f"\nSympy buchberger time: {elapsedTime:.6f} seconds")


Reduced Groebner has 4 basis polynomials:
x*z**2 + z - 1
-x*z + x + z**3
x**2 + z
y**2 - 1

Sympy buchberger time: 0.006344 seconds


# Our Implementation

In [6]:
from sympy import (symbols, Poly, Matrix, expand, sympify,lcm, gcd, Rational)

ORDER = 'grevlex'
x, y, z = symbols('x y z')
GENS = (x, y, z)
#Polynomial Utilities
def to_poly(f, gens=GENS):
  """Convert an expression or Poly to Poly."""
  if isinstance(f, Poly):
    if f.gens == tuple(gens):
      return f
    return Poly(f.as_expr(), *gens, domain='QQ')
  return Poly(expand(sympify(f)), *gens, domain='QQ')
def to_expr(f):
  """Convert a Poly or expression to an expression."""
  return f.as_expr() if isinstance(f, Poly) else sympify(f)
def ordered_terms(f, gens=GENS):
  """
  Return polynomial terms (exponent_tuple, coefficient) in descending grevlex order.
  """
  p = to_poly(f, gens)
  return p.terms(order=ORDER)

def leading_monomial_tuple(f, gens=GENS):
  """Return the leading monomial as an exponent tuple."""
  terms = ordered_terms(f, gens)
  if not terms:
    return None
  return terms[0][0]

def leading_coefficient(f, gens=GENS):
  """Return the leading coefficient under grevlex."""
  terms = ordered_terms(f, gens)
  if not terms:
    return 0
  return terms[0][1]
def monomial_expr(exponents, gens=GENS):
    """Convert an exponent tuple into a monomial expression."""
    result = 1
    for gen, exponent in zip(gens, exponents):
        result *= gen**exponent
    return result
def leading_monomial(f, gens=GENS):
  """Return the leading monomial without its coefficient."""
  exponents = leading_monomial_tuple(f, gens)
  if exponents is None:
    return 0
  return monomial_expr(exponents, gens)
def leading_term(f, gens=GENS):
  """Return the leading term under grevlex."""
  terms = ordered_terms(f, gens)
  if not terms:
    return 0
  exponents, coefficient = terms[0]
  return coefficient * monomial_expr(exponents, gens)
def monomial_divides(a, b):
  """Test whether monomial a divides monomial b."""
  return (len(a) == len(b) and all(ai <= bi for ai, bi in zip(a, b)))
def monomial_lcm_tuple(a, b):
  """Compute the LCM of two monomials."""
  return tuple(max(ai, bi) for ai, bi in zip(a, b))
def monomial_gcd_tuple(a, b):
  """Compute the GCD of two monomials."""
  return tuple(min(ai, bi) for ai, bi in zip(a, b))
def monomial_quotient(a, b):
  """Return a/b when b divides a."""
  if not monomial_divides(b, a):
    raise ValueError("The second monomial must divide the first.")
  return tuple(ai - bi for ai, bi in zip(a, b))
def monomial_degree(a):
  """Total degree of a monomial."""
  return sum(a)
def grevlex_key(exponents):
  """Sort monomials in descending grevlex order."""
  return (-sum(exponents),*reversed(exponents))
def NormalizedLeadingTerms(F, gens=GENS):
  """Collect normalized leading monomials"""
  result = {}
  for f in F:
    p = to_poly(f, gens)
    if p.is_zero:
      continue
    lm = leading_monomial(p, gens)
    result[lm] = 1
  return result

def LeadingTerms(F, gens=GENS):
  """Return dict of leading monomials in grelex."""
  result = {}
  for f in F:
    p = to_poly(f, gens)
    if p.is_zero:
      continue
    result[leading_monomial(p, gens)] = 1
  return result

def LeadingMonomials(F, gens=GENS):
  """Return leading monomials in grevlex"""
  result = []
  for f in F:
    p = to_poly(f, gens)
    if p.is_zero:
      continue
    lm = leading_monomial(p, gens)
    if lm not in result:
      result.append(lm)
  return result
def GetAllMonomials(f, gens=GENS):
  """Return terms in descending grevlex"""
  p = to_poly(f, gens)
  return [coefficient * monomial_expr(exponents, gens) for exponents, coefficient in ordered_terms(p, gens)]
def maCols(sHalf, gens=GENS):
  """Collect monomials tath aren't constants."""
  cols = set()
  for f in sHalf:
    p = to_poly(f, gens)
    if p.is_zero:
      continue
    for exponents, _ in ordered_terms(p, gens):
      if sum(exponents) > 0:
        cols.add(exponents)
  ordered = sorted(cols, key=grevlex_key)
  return [monomial_expr(m, gens) for m in ordered]
def AscOrdLT(F, gens=GENS):
  """Sort ascending grevlex"""
  result = []
  for f in F:
    p = to_poly(f, gens)
    if not p.is_zero:
      result.append(p)
  return sorted(result,key=lambda f: tuple(-v for v in grevlex_key(leading_monomial_tuple(f, gens))))
def multiVarDiv(f, G, gens=GENS):
  """Multivariable polynomial division using grevlext that return (quotients, remainder)"""
  f = to_poly(f, gens)
  G = [to_poly(g, gens)for g in G if not to_poly(g, gens).is_zero]
  p = f
  remainder = Poly(0, *gens, domain='QQ')
  quotients = [Poly(0, *gens, domain='QQ')for _ in G]
  while not p.is_zero:
    p_exp, p_coeff = ordered_terms(p, gens)[0]
    reduced = False
    for i, g in enumerate(G):
      g_exp, g_coeff = ordered_terms(g, gens)[0]
      if monomial_divides(g_exp, p_exp):
        q_exp = monomial_quotient(p_exp, g_exp)
        q_expr = (p_coeff / g_coeff * monomial_expr(q_exp, gens))
        q = Poly(q_expr, *gens, domain='QQ')
        quotients[i] += q
        p -= q * g
        reduced = True
        break
    if not reduced:
      term_expr = p_coeff * monomial_expr(p_exp, gens)
      term = Poly(term_expr, *gens, domain='QQ')
      remainder += term
      p -= term
    p = Poly(expand(p.as_expr()), *gens, domain='QQ')
  return quotients, remainder
# SPolynomials, SHalves
def SPolynomial(f, g, gens=GENS):
  """S-polynomial using grevlex leading terms."""
  f = to_poly(f, gens)
  g = to_poly(g, gens)
  if f.is_zero or g.is_zero:
    return Poly(0, *gens, domain='QQ')
  lm_f = leading_monomial_tuple(f, gens)
  lm_g = leading_monomial_tuple(g, gens)
  lc_f = leading_coefficient(f, gens)
  lc_g = leading_coefficient(g, gens)
  lcm_exp = monomial_lcm_tuple(lm_f, lm_g)
  qf = monomial_quotient(lcm_exp, lm_f)
  qg = monomial_quotient(lcm_exp, lm_g)
  expr = (monomial_expr(qf, gens) * f.as_expr() / lc_f - monomial_expr(qg, gens) * g.as_expr() / lc_g)
  return Poly(expand(expr), *gens, domain='QQ')

def SHalf(f, g, gens=GENS):
  """ S-half via the leading monomials """
  f = to_poly(f, gens)
  g = to_poly(g, gens)
  if f.is_zero or g.is_zero:
    return Poly(0, *gens, domain='QQ')
  lm_f = leading_monomial_tuple(f, gens)
  lm_g = leading_monomial_tuple(g, gens)
  lcm_exp = monomial_lcm_tuple(lm_f, lm_g)
  quotient_exp = monomial_quotient(lcm_exp, lm_f)
  return Poly(expand(monomial_expr(quotient_exp, gens) * f.as_expr()),*gens,domain='QQ')

def allS(sHalf, B, gens=GENS):
  """S-halves for critical pair"""
  result = []
  for i, j, *_ in B:
    if sHalf[i].is_zero or sHalf[j].is_zero:
      continue
    result.append(SHalf(sHalf[i], sHalf[j], gens))
    result.append(SHalf(sHalf[j], sHalf[i], gens))
  return result

def interred(F, gens=GENS):
  basis = []
  for f in F:
    p = to_poly(f, gens)
    if not p.is_zero and p not in basis:
      basis.append(p)
  if not basis:
    return []
  while True:
    old_leads = {leading_monomial_tuple(f, gens) for f in basis }
    new_basis = []
    for i, f in enumerate(basis):
      divisors = [g for j, g in enumerate(basis)if i != j]
      _, remainder = multiVarDiv(f, divisors, gens)
      if not remainder.is_zero:
        new_basis.append(remainder.monic())
    #Remove duplicates
    unique_basis = []
    for f in new_basis:
      if f not in unique_basis and -f not in unique_basis:
        unique_basis.append(f)
    basis = AscOrdLT(unique_basis, gens)
    new_leads = {leading_monomial_tuple(f, gens)for f in basis}
    if old_leads == new_leads:
      break
    if not basis:
      break
  return AscOrdLT([f.monic() for f in basis if not f.is_zero],gens)


def macaulay(F, gens=GENS):
    """Construc Macaulay matrix """
    polys = [to_poly(f, gens) for f in F]
    if not polys:
      return Matrix.zeros(0, 0), {}
    cols = maCols(polys, gens)
    colInd = {monomial: i for i, monomial in enumerate(cols)}
    needs_constant = any(any(sum(exponents) == 0 for exponents, _ in ordered_terms(p, gens)) for p in polys)
    ncols = len(cols) + int(needs_constant)
    matr = Matrix.zeros(len(polys), ncols)
    for i, p in enumerate(polys):
      for exponents, coefficient in ordered_terms(p, gens):
        monomial = monomial_expr(exponents, gens)
        if sum(exponents) == 0:
          if needs_constant:
            matr[i, len(cols)] += coefficient
        else:
          matr[i, colInd[monomial]] += coefficient
    return matr, colInd

def symPre(sHalf, G, gens=GENS):
  """Symbolic preprocessing"""
  sHalf = [to_poly(f, gens) for f in sHalf if not to_poly(f, gens).is_zero]
  G = [to_poly(g, gens) for g in G if not to_poly(g, gens).is_zero]
  if not sHalf:
    return []
  proc = {leading_monomial_tuple(f, gens) for f in sHalf}
  check = {exponents for f in sHalf for exponents, _ in ordered_terms(f, gens)if sum(exponents) > 0}
  com = sorted(check, key=grevlex_key)
  while com:
      current = com.pop()
      if current in proc:
        continue
      proc.add(current)
      for g in G:
        lm_g = leading_monomial_tuple(g, gens)
        if monomial_divides(lm_g, current):
          quotient_exp = monomial_quotient(current, lm_g)
          ele = Poly(expand(monomial_expr(quotient_exp, gens)* g.as_expr()),*gens,domain='QQ')
          sHalf.append(ele)
          for exponents, _ in ordered_terms(ele, gens):
              if(sum(exponents) > 0 and exponents not in proc):
                com.append(exponents)
          break
  return sHalf

def Red(L, G, ver=0, gens=GENS):
  """Reduction logic"""
  if ver == 1:
    print('Reduction inputs:', L, G)
  M = symPre(L, G, gens)
  if not M:
    return [], 0
  M_matrix, colInd = macaulay(M, gens)
  if ver == 1:
    print('Macaulay matrix:')
    print(M_matrix)
  invCols = {v: k for k, v in colInd.items()}
  if colInd:
      first_exp = leading_monomial_tuple(next(iter(colInd.keys())),gens)
      SD = monomial_degree(first_exp)
  else:
      SD = 0
  N, pivots = M_matrix.rref()
  if ver == 1:
      print('Reduced row-echelon form:')
      print(N)
  lmM = LeadingTerms(M, gens)
  Ftildeplus = []
  for i in range(N.rows):
    first_nonzero = None
    for j in range(N.cols):
      if N[i, j] != 0:
        first_nonzero = j
        break
    if first_nonzero is None:
      continue
    if first_nonzero < len(colInd):
      monomial = invCols[first_nonzero]
      if monomial in lmM:
        continue
    expr = 0
    for j in range(len(colInd)):
      if N[i, j] != 0:
        expr += invCols[j] * N[i, j]
    if N.cols > len(colInd):
      expr += N[i, len(colInd)]
    expr = expand(expr)
    if expr != 0:
      Ftildeplus.append(Poly(expr, *gens, domain='QQ'))
  if ver == 1:
    print('New polynomials:', Ftildeplus)
  return Ftildeplus, SD

def F4(F, ver=0, gens=None):
  if not F:
    return []
  if gens is None:
    if isinstance(F[0], Poly):
      gens = F[0].gens
    else:
      symbols_set = set()
      for f in F:
        symbols_set.update(sympify(f).free_symbols)
      gens = tuple(sorted(symbols_set, key=lambda s: s.name))
  gens = tuple(gens)
  F = [to_poly(f, gens)for f in F if not to_poly(f, gens).is_zero]
  if not F:
    return []
  G = interred(F, gens)
  B = []
  for i in range(len(G)):
    for j in range(i + 1, len(G)):
      lm_i = leading_monomial_tuple(G[i], gens)
      lm_j = leading_monomial_tuple(G[j], gens)
      degree = monomial_degree(monomial_lcm_tuple(lm_i, lm_j))
      B.append([i, j, degree])
  stepDeg = []
  count = 0
  while B:
    smallest = min(pair[2] for pair in B)
    selected = [pair for pair in B if pair[2] == smallest]
    B = [pair for pair in B if pair[2] != smallest]
    if ver == 1:
      print(f'\nIteration {count + 1}')
      print('Selected pairs:', selected)
      print('Current basis:', G)
    L = allS(G, selected, gens)
    old_len = len(G)
    Ftp, SD = Red(L, G, ver, gens)
    stepDeg.append(SD)
    for f in Ftp:
      _, remainder = multiVarDiv(f, G, gens)

      if not remainder.is_zero:
        remainder = remainder.monic()
        if(remainder not in G and -remainder not in G):
          G.append(remainder)

    for j in range(old_len):
      for i in range(old_len, len(G)):
        lm_j = leading_monomial_tuple(G[j], gens)
        lm_i = leading_monomial_tuple(G[i], gens)
        degree = monomial_degree(monomial_lcm_tuple(lm_j, lm_i))
        B.append([j, i, degree])
    count += 1
  G = interred(G, gens)
  if ver == 1:
      print('\nStep degrees:', stepDeg)
      print('\nFinal basis:')
      for f in G:
          print(f.as_expr())
  return G



x, y, z = symbols('x y z')
gens = (x, y, z)

F = [
    Poly(x**3 + y**2 + x*z - 1, *gens, domain='QQ'),
    Poly(x**2 + y**2 + z - 1, *gens, domain='QQ'),
    Poly(y**2*z + x*z**2 - 1, *gens, domain='QQ')
]

G = F4(F, ver=1, gens=gens)




Iteration 1
Selected pairs: [[0, 2, 4], [1, 2, 4]]
Current basis: [Poly(x**2 + y**2 + z - 1, x, y, z, domain='QQ'), Poly(x*z**2 + y**2*z - 1, x, y, z, domain='QQ'), Poly(x*y**2 - x - y**2 + 1, x, y, z, domain='QQ')]
Reduction inputs: [Poly(x**2*y**2 + y**4 + y**2*z - y**2, x, y, z, domain='QQ'), Poly(x**2*y**2 - x**2 - x*y**2 + x, x, y, z, domain='QQ'), Poly(x**2*z**2 + x*y**2*z - x, x, y, z, domain='QQ'), Poly(x*y**2*z - x*z - y**2*z + z, x, y, z, domain='QQ')] [Poly(x**2 + y**2 + z - 1, x, y, z, domain='QQ'), Poly(x*z**2 + y**2*z - 1, x, y, z, domain='QQ'), Poly(x*y**2 - x - y**2 + 1, x, y, z, domain='QQ')]
Macaulay matrix:
Matrix([[1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, -1, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, -1, 0, 0, 0, -1, 0, 0, 0, 1, 0, 0], [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0], [0, 0, 1, 0, 0, 0, 0, -1, 0, 0, 0, 0, -1, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, -1], [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, -1], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 

In [2]:
from sympy import (
    symbols, Poly, Matrix, expand, sympify,
    lcm, gcd, Rational
)

# ============================================================
# GLOBAL MONOMIAL ORDER
# ============================================================

ORDER = 'grevlex'

x, y, z = symbols('x y z')
GENS = (x, y, z)


# ============================================================
# BASIC POLYNOMIAL UTILITIES
# ============================================================

def to_poly(f, gens=GENS):
    """Convert an expression or Poly to Poly."""
    if isinstance(f, Poly):
        if f.gens == tuple(gens):
            return f
        return Poly(f.as_expr(), *gens, domain='QQ')

    return Poly(expand(sympify(f)), *gens, domain='QQ')


def to_expr(f):
    """Convert a Poly or expression to an expression."""
    return f.as_expr() if isinstance(f, Poly) else sympify(f)


def ordered_terms(f, gens=GENS):
    """
    Return polynomial terms in descending grevlex order.

    Each item is (exponent_tuple, coefficient).
    """
    p = to_poly(f, gens)
    return p.terms(order=ORDER)


def leading_monomial_tuple(f, gens=GENS):
    """Return the leading monomial as an exponent tuple."""
    terms = ordered_terms(f, gens)

    if not terms:
        return None

    return terms[0][0]


def leading_coefficient(f, gens=GENS):
    """Return the leading coefficient under grevlex."""
    terms = ordered_terms(f, gens)

    if not terms:
        return 0

    return terms[0][1]


def monomial_expr(exponents, gens=GENS):
    """Convert an exponent tuple into a monomial expression."""
    result = 1

    for gen, exponent in zip(gens, exponents):
        result *= gen**exponent

    return result


def leading_monomial(f, gens=GENS):
    """Return the leading monomial without its coefficient."""
    exponents = leading_monomial_tuple(f, gens)

    if exponents is None:
        return 0

    return monomial_expr(exponents, gens)


def leading_term(f, gens=GENS):
    """Return the leading term under grevlex."""
    terms = ordered_terms(f, gens)

    if not terms:
        return 0

    exponents, coefficient = terms[0]

    return coefficient * monomial_expr(exponents, gens)


def monomial_divides(a, b):
    """
    Test whether monomial a divides monomial b.

    Both monomials are exponent tuples.
    """
    return (
        len(a) == len(b)
        and all(ai <= bi for ai, bi in zip(a, b))
    )


def monomial_lcm_tuple(a, b):
    """Compute the LCM of two monomials."""
    return tuple(max(ai, bi) for ai, bi in zip(a, b))


def monomial_gcd_tuple(a, b):
    """Compute the GCD of two monomials."""
    return tuple(min(ai, bi) for ai, bi in zip(a, b))


def monomial_quotient(a, b):
    """
    Return a/b when b divides a.
    """
    if not monomial_divides(b, a):
        raise ValueError("The second monomial must divide the first.")

    return tuple(ai - bi for ai, bi in zip(a, b))


def monomial_degree(a):
    """Total degree of a monomial."""
    return sum(a)


def grevlex_key(exponents):
    """
    Sort monomials in descending grevlex order.

    Python sorts keys in ascending order, so this key reverses
    the desired monomial ordering.
    """
    return (
        -sum(exponents),
        *reversed(exponents)
    )


# ============================================================
# S-POLYNOMIALS AND S-HALVES
# ============================================================

def SPolynomial(f, g, gens=GENS):
    """
    Compute the standard S-polynomial using grevlex leading terms.
    """
    f = to_poly(f, gens)
    g = to_poly(g, gens)

    if f.is_zero or g.is_zero:
        return Poly(0, *gens, domain='QQ')

    lm_f = leading_monomial_tuple(f, gens)
    lm_g = leading_monomial_tuple(g, gens)

    lc_f = leading_coefficient(f, gens)
    lc_g = leading_coefficient(g, gens)

    lcm_exp = monomial_lcm_tuple(lm_f, lm_g)
    qf = monomial_quotient(lcm_exp, lm_f)
    qg = monomial_quotient(lcm_exp, lm_g)

    expr = (
        monomial_expr(qf, gens) * f.as_expr() / lc_f
        - monomial_expr(qg, gens) * g.as_expr() / lc_g
    )

    return Poly(expand(expr), *gens, domain='QQ')


def SHalf(f, g, gens=GENS):
    """
    Compute a directed S-half using the leading monomials.

    This is a polynomial multiple of f whose leading monomial
    is the LCM of the leading monomials of f and g.
    """
    f = to_poly(f, gens)
    g = to_poly(g, gens)

    if f.is_zero or g.is_zero:
        return Poly(0, *gens, domain='QQ')

    lm_f = leading_monomial_tuple(f, gens)
    lm_g = leading_monomial_tuple(g, gens)

    lcm_exp = monomial_lcm_tuple(lm_f, lm_g)
    quotient_exp = monomial_quotient(lcm_exp, lm_f)

    return Poly(
        expand(monomial_expr(quotient_exp, gens) * f.as_expr()),
        *gens,
        domain='QQ'
    )


# ============================================================
# LEADING TERMS AND MONOMIAL COLLECTION
# ============================================================

def NormalizedLeadingTerms(F, gens=GENS):
    """Collect distinct leading monomials."""
    result = {}

    for f in F:
        p = to_poly(f, gens)

        if p.is_zero:
            continue

        lm = leading_monomial(p, gens)
        result[lm] = 1

    return result


def LeadingMonomials(F, gens=GENS):
    """Return distinct leading monomials in grevlex order."""
    result = []

    for f in F:
        p = to_poly(f, gens)

        if p.is_zero:
            continue

        lm = leading_monomial(p, gens)

        if lm not in result:
            result.append(lm)

    return result


def GetAllMonomials(f, gens=GENS):
    """
    Return individual terms in descending grevlex order.
    """
    p = to_poly(f, gens)

    return [
        coefficient * monomial_expr(exponents, gens)
        for exponents, coefficient in ordered_terms(p, gens)
    ]


def maCols(sHalf, gens=GENS):
    """
    Collect distinct nonconstant monomials appearing in polynomials.
    """
    cols = set()

    for f in sHalf:
        p = to_poly(f, gens)

        if p.is_zero:
            continue

        for exponents, _ in ordered_terms(p, gens):
            if sum(exponents) > 0:
                cols.add(exponents)

    # Columns are ordered by descending grevlex.
    ordered = sorted(cols, key=grevlex_key)

    return [monomial_expr(m, gens) for m in ordered]


def AscOrdLT(F, gens=GENS):
    """
    Sort polynomials by ascending grevlex order of their leading
    monomials.
    """
    result = []

    for f in F:
        p = to_poly(f, gens)

        if not p.is_zero:
            result.append(p)

    return sorted(
        result,
        key=lambda f: tuple(
            -v for v in grevlex_key(
                leading_monomial_tuple(f, gens)
            )
        )
    )


# ============================================================
# MULTIVARIABLE DIVISION
# ============================================================

def multiVarDiv(f, G, gens=GENS):
    """
    Multivariable polynomial division using grevlex leading terms.

    Returns:
        (quotients, remainder)
    """
    f = to_poly(f, gens)

    G = [
        to_poly(g, gens)
        for g in G
        if not to_poly(g, gens).is_zero
    ]

    p = f
    remainder = Poly(0, *gens, domain='QQ')
    quotients = [
        Poly(0, *gens, domain='QQ')
        for _ in G
    ]

    while not p.is_zero:
        p_exp, p_coeff = ordered_terms(p, gens)[0]
        reduced = False

        for i, g in enumerate(G):
            g_exp, g_coeff = ordered_terms(g, gens)[0]

            if monomial_divides(g_exp, p_exp):
                q_exp = monomial_quotient(p_exp, g_exp)

                q_expr = (
                    p_coeff / g_coeff
                    * monomial_expr(q_exp, gens)
                )

                q = Poly(q_expr, *gens, domain='QQ')

                quotients[i] += q
                p -= q * g
                reduced = True
                break

        if not reduced:
            term_expr = p_coeff * monomial_expr(p_exp, gens)
            term = Poly(term_expr, *gens, domain='QQ')

            remainder += term
            p -= term

        p = Poly(expand(p.as_expr()), *gens, domain='QQ')

    return quotients, remainder


# ============================================================
# INTERREDUCTION
# ============================================================

def interred(F, gens=GENS):
    """
    Interreduce a polynomial list using grevlex leading terms.
    """
    basis = []

    for f in F:
        p = to_poly(f, gens)

        if not p.is_zero and p not in basis:
            basis.append(p)

    if not basis:
        return []

    while True:
        old_leads = {
            leading_monomial_tuple(f, gens)
            for f in basis
        }

        new_basis = []

        for i, f in enumerate(basis):
            divisors = [
                g for j, g in enumerate(basis)
                if i != j
            ]

            _, remainder = multiVarDiv(f, divisors, gens)

            if not remainder.is_zero:
                new_basis.append(remainder.monic())

        # Remove duplicates.
        unique_basis = []

        for f in new_basis:
            if f not in unique_basis and -f not in unique_basis:
                unique_basis.append(f)

        basis = AscOrdLT(unique_basis, gens)

        new_leads = {
            leading_monomial_tuple(f, gens)
            for f in basis
        }

        if old_leads == new_leads:
            break

        if not basis:
            break

    return AscOrdLT(
        [f.monic() for f in basis if not f.is_zero],
        gens
    )


# ============================================================
# CRITICAL PAIRS
# ============================================================

def allS(sHalf, B, gens=GENS):
    """
    Compute directed S-halves for selected critical pairs.
    """
    result = []

    for i, j, *_ in B:
        if sHalf[i].is_zero or sHalf[j].is_zero:
            continue

        result.append(SHalf(sHalf[i], sHalf[j], gens))
        result.append(SHalf(sHalf[j], sHalf[i], gens))

    return result


# ============================================================
# MACAULAY MATRIX
# ============================================================

def macaulay(F, gens=GENS):
    """
    Construct a Macaulay matrix.

    Nonconstant monomial columns are ordered in descending grevlex.
    The constant column, when needed, is last.
    """
    polys = [to_poly(f, gens) for f in F]

    if not polys:
        return Matrix.zeros(0, 0), {}

    cols = maCols(polys, gens)

    colInd = {
        monomial: i
        for i, monomial in enumerate(cols)
    }

    needs_constant = any(
        any(sum(exponents) == 0 for exponents, _ in ordered_terms(p, gens))
        for p in polys
    )

    ncols = len(cols) + int(needs_constant)
    matr = Matrix.zeros(len(polys), ncols)

    for i, p in enumerate(polys):
        for exponents, coefficient in ordered_terms(p, gens):
            monomial = monomial_expr(exponents, gens)

            if sum(exponents) == 0:
                if needs_constant:
                    matr[i, len(cols)] += coefficient
            else:
                matr[i, colInd[monomial]] += coefficient

    return matr, colInd


# ============================================================
# SYMBOLIC PREPROCESSING
# ============================================================

def symPre(sHalf, G, gens=GENS):
    """
    Symbolic preprocessing using grevlex leading monomials.
    """
    sHalf = [
        to_poly(f, gens)
        for f in sHalf
        if not to_poly(f, gens).is_zero
    ]

    G = [
        to_poly(g, gens)
        for g in G
        if not to_poly(g, gens).is_zero
    ]

    if not sHalf:
        return []

    proc = {
        leading_monomial_tuple(f, gens)
        for f in sHalf
    }

    check = {
        exponents
        for f in sHalf
        for exponents, _ in ordered_terms(f, gens)
        if sum(exponents) > 0
    }

    com = sorted(check, key=grevlex_key)

    while com:
        current = com.pop()

        if current in proc:
            continue

        proc.add(current)

        for g in G:
            lm_g = leading_monomial_tuple(g, gens)

            if monomial_divides(lm_g, current):
                quotient_exp = monomial_quotient(current, lm_g)

                ele = Poly(
                    expand(
                        monomial_expr(quotient_exp, gens)
                        * g.as_expr()
                    ),
                    *gens,
                    domain='QQ'
                )

                sHalf.append(ele)

                # Add any newly encountered monomials.
                for exponents, _ in ordered_terms(ele, gens):
                    if (
                        sum(exponents) > 0
                        and exponents not in proc
                    ):
                        com.append(exponents)

                break

    return sHalf


# ============================================================
# LEADING MONOMIAL DICTIONARY
# ============================================================

def leadingTerms(F, gens=GENS):
    """
    Return a dictionary of leading monomials under grevlex.
    """
    result = {}

    for f in F:
        p = to_poly(f, gens)

        if p.is_zero:
            continue

        result[leading_monomial(p, gens)] = 1

    return result


# ============================================================
# F4 REDUCTION
# ============================================================

def Red(L, G, ver=0, gens=GENS):
    """
    Symbolic preprocessing and Macaulay matrix reduction.

    This retains the structure of your original implementation.
    """
    if ver == 1:
        print('Reduction inputs:', L, G)

    M = symPre(L, G, gens)

    if not M:
        return [], 0

    M_matrix, colInd = macaulay(M, gens)

    if ver == 1:
        print('Macaulay matrix:')
        print(M_matrix)

    invCols = {v: k for k, v in colInd.items()}

    if colInd:
        first_exp = leading_monomial_tuple(
            next(iter(colInd.keys())),
            gens
        )
        SD = monomial_degree(first_exp)
    else:
        SD = 0

    N, pivots = M_matrix.rref()

    if ver == 1:
        print('Reduced row-echelon form:')
        print(N)

    lmM = leadingTerms(M, gens)
    Ftildeplus = []

    for i in range(N.rows):
        first_nonzero = None

        for j in range(N.cols):
            if N[i, j] != 0:
                first_nonzero = j
                break

        if first_nonzero is None:
            continue

        # Skip the row if its leading monomial is already represented
        # by the input/preprocessed leading monomials.
        if first_nonzero < len(colInd):
            monomial = invCols[first_nonzero]

            if monomial in lmM:
                continue

        expr = 0

        for j in range(len(colInd)):
            if N[i, j] != 0:
                expr += invCols[j] * N[i, j]

        if N.cols > len(colInd):
            expr += N[i, len(colInd)]

        expr = expand(expr)

        if expr != 0:
            Ftildeplus.append(
                Poly(expr, *gens, domain='QQ')
            )

    if ver == 1:
        print('New polynomials:', Ftildeplus)

    return Ftildeplus, SD


# ============================================================
# F4 ALGORITHM
# ============================================================

def F4(F, ver=0, gens=None):
    """
    F4-style Gröbner basis computation with grevlex monomial ordering.

    Note:
        This preserves the overall structure of the supplied code.
        See the explanation below concerning full F4 correctness.
    """
    if not F:
        return []

    if gens is None:
        if isinstance(F[0], Poly):
            gens = F[0].gens
        else:
            symbols_set = set()

            for f in F:
                symbols_set.update(sympify(f).free_symbols)

            gens = tuple(sorted(symbols_set, key=lambda s: s.name))

    gens = tuple(gens)

    F = [
        to_poly(f, gens)
        for f in F
        if not to_poly(f, gens).is_zero
    ]

    if not F:
        return []

    G = interred(F, gens)

    B = []

    for i in range(len(G)):
        for j in range(i + 1, len(G)):
            lm_i = leading_monomial_tuple(G[i], gens)
            lm_j = leading_monomial_tuple(G[j], gens)

            degree = monomial_degree(
                monomial_lcm_tuple(lm_i, lm_j)
            )

            B.append([i, j, degree])

    stepDeg = []
    count = 0

    while B:
        smallest = min(pair[2] for pair in B)

        selected = [
            pair for pair in B
            if pair[2] == smallest
        ]

        B = [
            pair for pair in B
            if pair[2] != smallest
        ]

        if ver == 1:
            print(f'\nIteration {count + 1}')
            print('Selected pairs:', selected)
            print('Current basis:', G)

        L = allS(G, selected, gens)

        old_len = len(G)

        Ftp, SD = Red(L, G, ver, gens)
        stepDeg.append(SD)

        for f in Ftp:
            _, remainder = multiVarDiv(f, G, gens)

            if not remainder.is_zero:
                remainder = remainder.monic()

                if (
                    remainder not in G
                    and -remainder not in G
                ):
                    G.append(remainder)

        for j in range(old_len):
            for i in range(old_len, len(G)):
                lm_j = leading_monomial_tuple(G[j], gens)
                lm_i = leading_monomial_tuple(G[i], gens)

                degree = monomial_degree(
                    monomial_lcm_tuple(lm_j, lm_i)
                )

                B.append([j, i, degree])

        count += 1

    G = interred(G, gens)

    if ver == 1:
        print('\nStep degrees:', stepDeg)
        print('\nFinal basis:')

        for f in G:
            print(f.as_expr())

    return G


# ============================================================
# TEST EXAMPLE
# ============================================================

x, y, z = symbols('x y z')
gens = (x, y, z)

F = [
    Poly(x**3 + y**2 + x*z - 1, *gens, domain='QQ'),
    Poly(x**2 + y**2 + z - 1, *gens, domain='QQ'),
    Poly(y**2*z + x*z**2 - 1, *gens, domain='QQ')
]

G = F4(F, ver=1, gens=gens)

print('\nFinal basis:')
for f in G:
    print(f.as_expr())



Iteration 1
Selected pairs: [[0, 2, 4], [1, 2, 4]]
Current basis: [Poly(x**2 + y**2 + z - 1, x, y, z, domain='QQ'), Poly(x*z**2 + y**2*z - 1, x, y, z, domain='QQ'), Poly(x*y**2 - x - y**2 + 1, x, y, z, domain='QQ')]
Reduction inputs: [Poly(x**2*y**2 + y**4 + y**2*z - y**2, x, y, z, domain='QQ'), Poly(x**2*y**2 - x**2 - x*y**2 + x, x, y, z, domain='QQ'), Poly(x**2*z**2 + x*y**2*z - x, x, y, z, domain='QQ'), Poly(x*y**2*z - x*z - y**2*z + z, x, y, z, domain='QQ')] [Poly(x**2 + y**2 + z - 1, x, y, z, domain='QQ'), Poly(x*z**2 + y**2*z - 1, x, y, z, domain='QQ'), Poly(x*y**2 - x - y**2 + 1, x, y, z, domain='QQ')]
Macaulay matrix:
Matrix([[1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, -1, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, -1, 0, 0, 0, -1, 0, 0, 0, 1, 0, 0], [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0], [0, 0, 1, 0, 0, 0, 0, -1, 0, 0, 0, 0, -1, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, -1], [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, -1], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 